## Churn Prediction: Exploratory Data Analysis

The actual notebooks, contain the EDA for the Ecommerce Customer Behavior Dataset from [kaggle](https://www.kaggle.com/datasets/jenifer201095/ecommerce-customer-behavior-dataset?resource=download "Go to dataset").

### Initial setup

In [27]:
## Load working libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

In [28]:

## Load dataset
### Set data path
DATA_PATH = "../data/raw/behavior.csv"

### Load the dataset from data path
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print("✨ Environment and data successfully connected in Positron! ✨")
    print(f"📊 Dataset consists of {df.shape[0]} rows and {df.shape[1]} features.\n")
    display(df.head())
else:
    print(f"❌ Error: Dataset not found at {DATA_PATH}")

✨ Environment and data successfully connected in Positron! ✨
📊 Dataset consists of 25000 rows and 8 features.



,customer_id,avg_session_time,pages_per_session,cart_abandon_rate,return_rate,support_tickets,review_score,behavior_churn_signal
0,CUST000001,8.12,21,0.95,0.37,1,3.39,1
1,CUST000002,3.96,9,0.16,0.03,3,4.46,1
2,CUST000003,12.42,24,0.71,0.01,1,4.88,0
3,CUST000004,16.82,18,0.21,0.09,1,1.73,0
4,CUST000005,6.78,2,0.52,0.22,0,2.16,1


### Basic data integrity check

In [29]:
## Missing values and Target Variable balance check
print("=== 1. Data Types and Missing Values ===")
print(df.info())

print("\n=== 2. Target Variable Balance (Churn Signal) ===")
churn_counts = df['behavior_churn_signal'].value_counts()
churn_percentages = df['behavior_churn_signal'].value_counts(normalize=True) * 100

for val in [0, 1]:
    print(f"Class {val} (Non-Churn/Churn): {churn_counts[val]} rows ({churn_percentages[val]:.2f}%)")

=== 1. Data Types and Missing Values ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            25000 non-null  object 
 1   avg_session_time       25000 non-null  float64
 2   pages_per_session      25000 non-null  int64  
 3   cart_abandon_rate      25000 non-null  float64
 4   return_rate            25000 non-null  float64
 5   support_tickets        25000 non-null  int64  
 6   review_score           25000 non-null  float64
 7   behavior_churn_signal  25000 non-null  int64  
dtypes: float64(4), int64(3), object(1)
memory usage: 1.5+ MB
None

=== 2. Target Variable Balance (Churn Signal) ===
Class 0 (Non-Churn/Churn): 16388 rows (65.55%)
Class 1 (Non-Churn/Churn): 8612 rows (34.45%)


In [30]:
## Statistical summary
print("\n=== 3. Statistical Summary ===")
display(df.describe())


=== 3. Statistical Summary ===


,avg_session_time,pages_per_session,cart_abandon_rate,return_rate,support_tickets,review_score,behavior_churn_signal
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,10.468978,13.080840,0.497954,0.250166,5.011960,3.005011,0.344480
std,5.472970,7.225421,0.287699,0.144221,3.171722,1.157417,0.475208
min,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,5.690000,7.000000,0.250000,0.130000,2.000000,2.010000,0.000000
50%,10.490000,13.000000,0.500000,0.250000,5.000000,3.010000,0.000000
75%,15.200000,19.000000,0.750000,0.380000,8.000000,4.010000,1.000000
max,20.000000,25.000000,1.000000,0.500000,10.000000,5.000000,1.000000


### Features Engieneering

In [31]:
## Developing new features
# 1. Frustration Index (High tickets + Low review scores)
df['frustration_index'] = df['support_tickets'] / (df['review_score'] + 0.1)

# 2. Value-to-Bail Ratio (Browsing depth vs. checkout abandonment)
df['value_to_bail'] = df['pages_per_session'] * (1 - df['cart_abandon_rate'])

# 3. Return Defect Risk (Logistical hassle interaction)
df['return_defect_risk'] = df['return_rate'] * df['support_tickets']

print("=== 4. Feature engineering complete! New columns added. ===")
print(df[['frustration_index', 'value_to_bail', 'return_defect_risk']].head())

=== 4. Feature engineering complete! New columns added. ===
   frustration_index  value_to_bail  return_defect_risk
0           0.286533           1.05                0.37
1           0.657895           7.56                0.09
2           0.200803           6.96                0.01
3           0.546448          14.22                0.09
4           0.000000           0.96                0.00


Since the target variable is moderately imbalance, there is not necessary applied sampling methodologies. then, we proceed to train the baseline model.

### Baseline Model training (Logistic Regression)

In [32]:
## Step 1: Split target from features
X = df.drop(columns = ['customer_id', 'behavior_churn_signal']) #dropping customer_id an target
y = df['behavior_churn_signal'] #creating target dataset

print('=== Step 1: Splitting features and target ===')
print(f'X shape: {X.shape} features \ny shape: {y.shape} target')

=== Step 1: Splitting features and target ===
X shape: (25000, 9) features 
y shape: (25000,) target


In [33]:
## Step 2: Split train and test using the stratify of y (keep 65/35 ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

print('=== Step 2: Splitting train and test with stratifying ===')
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'y_train: {y_train.shape} | y_test: {y_test.shape}')

print(f'\nTrain Churn Ratio: {y_train.value_counts(normalize=True)[1]:.2%}')
print(f'Test Churn Ratio: {y_test.value_counts(normalize=True)[1]:.2%}')

=== Step 2: Splitting train and test with stratifying ===
X_train: (20000, 9) | X_test: (5000, 9)
y_train: (20000,) | y_test: (5000,)

Train Churn Ratio: 34.45%
Test Churn Ratio: 34.44%


In [34]:
## Step 3: Scaling features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

train_means = np.mean(X_train_scaled, axis=0)
train_stds = np.std(X_train_scaled, axis=0)

print("=== Step 3: Scaling and normalization train features ===")
for i in range(6):
    print(f"   Feature {i} -> Mean: {train_means[i]:.4f} (Target: ~0) | Std Dev: {train_stds[i]:.4f} (Target: 1)")

=== Step 3: Scaling and normalization train features ===
   Feature 0 -> Mean: -0.0000 (Target: ~0) | Std Dev: 1.0000 (Target: 1)
   Feature 1 -> Mean: -0.0000 (Target: ~0) | Std Dev: 1.0000 (Target: 1)
   Feature 2 -> Mean: 0.0000 (Target: ~0) | Std Dev: 1.0000 (Target: 1)
   Feature 3 -> Mean: 0.0000 (Target: ~0) | Std Dev: 1.0000 (Target: 1)
   Feature 4 -> Mean: -0.0000 (Target: ~0) | Std Dev: 1.0000 (Target: 1)
   Feature 5 -> Mean: -0.0000 (Target: ~0) | Std Dev: 1.0000 (Target: 1)


In [35]:
## Step 4: Training baseline model (logistic regression)
baseline_model = LogisticRegression(random_state=42)
baseline_model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [36]:
## Step 5: Baseline performance report
y_pred = baseline_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.92      0.78      3278
           1       0.55      0.18      0.27      1722

    accuracy                           0.67      5000
   macro avg       0.62      0.55      0.53      5000
weighted avg       0.64      0.67      0.61      5000



The model results show that the logistic regression is being effective at predicting customers who are not going to churn (68% precision), while it fails in 82% of the cases that are actually marked for churn. Likewise, the f1-score is well below the expected benchmark (75%). Therefore, the results will not generate added value for the business.

Therefore, we will evaluate a Random Forest model, which is a good option when the target variable is moderatly imbalanced.

### Evaluate contrast model (Random Forest)

In [37]:
## Step 6: Training contrast model (Random Forest)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [38]:
## Step 7: Contrast performance report
y_pred_rf = rf_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.74      0.64      0.69      3278
           1       0.46      0.57      0.51      1722

    accuracy                           0.62      5000
   macro avg       0.60      0.61      0.60      5000
weighted avg       0.64      0.62      0.63      5000



With the application of Random Forest, the performance metrics show effective improvements. The new model is capable of predicting more than half of the actual churn cases; additionally, the F1-score also shows an improvement, indicating that the model is becoming more precise.

To further fine-tune the prediction, we will tune the model's hyperparameters using GridSearchCV.

### Hyperparameter Tuning

In [39]:
## Step 8: Tuning the hyperparameters
### 1. Define the parameter space we want to explore
param_grid = {
    'n_estimators': [100, 200],          # Number of trees in the forest
    'max_depth': [10, 15, None],          # Deepest path allowed in a tree
    'min_samples_split': [2, 5],         # Minimum samples required to split a node
    'min_samples_leaf': [1, 2]           # Minimum samples required at a leaf node
}

### 2. Instantiate the base model with balanced weights
base_rf = RandomForestClassifier(random_state=42, class_weight='balanced')

### 3. Set up the Grid Search using 3-fold Cross-Validation 
# We optimize specifically for 'f1' since that's our business target metric!
grid_search = GridSearchCV(
    estimator=base_rf, 
    param_grid=param_grid, 
    cv=3, 
    scoring='f1', 
    n_jobs=-1, 
    verbose=1
)

### 4. Run the search on our scaled training data
grid_search.fit(X_train_scaled, y_train)

print("\n🏆 OPTIMIZATION COMPLETE 🏆")
print(f"🔥 Best Hyperparameters Found: {grid_search.best_params_}")

### 5. Evaluate the optimized model on our test data
best_rf_model = grid_search.best_estimator_
y_pred_tuned = best_rf_model.predict(X_test_scaled)

print("\n=== 📊 TUNED RANDOM FOREST PERFORMANCE REPORT ===")
print(classification_report(y_test, y_pred_tuned))

Fitting 3 folds for each of 24 candidates, totalling 72 fits

🏆 OPTIMIZATION COMPLETE 🏆
🔥 Best Hyperparameters Found: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}

=== 📊 TUNED RANDOM FOREST PERFORMANCE REPORT ===
              precision    recall  f1-score   support

           0       0.73      0.64      0.68      3278
           1       0.45      0.56      0.50      1722

    accuracy                           0.61      5000
   macro avg       0.59      0.60      0.59      5000
weighted avg       0.64      0.61      0.62      5000



Since tuning failed to significantly improve the performance metrics of the random forest, we will proceed to test a gradient boosting method, such as xgboost.

### Evaluate second contrast model (xgboost)

In [41]:
## step 9: Training new contrast model
xgb_tuned = XGBClassifier(
    n_estimators=300,          # More trees to let sequential learning complete
    max_depth=8,               # Deeper trees to catch complex interactions
    learning_rate=0.2,         # Faster learning step to adapt to skewed data
    scale_pos_weight=1.9,      # Keeping our balance handle for Class 1
    subsample=0.8,             # Use 80% of rows per tree to reduce overfitting
    colsample_bytree=0.8,      # Use 80% of features per tree to force feature exploration
    random_state=42,
    eval_metric='logloss'
)

xgb_tuned.fit(X_train_scaled, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [42]:
## step 10: Contrast performance report
y_pred_xgb = xgb_tuned.predict(X_test_scaled)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.70      0.73      0.71      3278
           1       0.43      0.40      0.41      1722

    accuracy                           0.61      5000
   macro avg       0.56      0.56      0.56      5000
weighted avg       0.60      0.61      0.61      5000

